### QM mechanistic descriptor extraction and appending to CSV file
Adjusted mechanism of the TM, including the dimers

##### Import statements

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

#### Input data

In [ ]:
full_data_path_QM= r"NEW-Reizmann_AIMNet_Gtotal_msrrho_20C_120C_all_species_new_reduced_all_methods.csv"
df_QM = pd.read_csv(full_data_path_QM)

full_data_path_reizman = r"Reizman_database - COMPLETE - BASELINE.xlsx"
df_reizman = pd.read_excel(full_data_path_reizman)

df_reactant9 = pd.read_csv(r"TM9_FINAL\TM9_FINAL\tables\TM9_Gtotal_msrrho_20C_120C_all_species_wide_kcalmol_plus8_species_plus5_SPE-FINAL.csv")

df_reactant8 = pd.read_csv(r"TM8_FINAL\TM8_FINAL\tables\TM8_Gtotal_msrrho_20C_120C_FINAL.csv")

df_BOH3 = pd.read_csv(r"BOH3_neutral_recalc\BOH3_neutral_recalc\tables\BOH3_Gtotal_msrrho_20C_120C_all_species_wide_kcalmol.csv")

df_ggw_reactions_all_species = pd.read_csv(r"Final_equilibria\Final_equilibria\FINAL_GGW_reactions_r2scan3c_RRHO_20C_120C_FROM_ALL_SPECIES.csv")

df_precat = pd.read_csv(r"final_precatalysts\final_precatalysts\tables\Precat_Gtotal_msrrho_20C_120C_all_species_wide_kcalmol.csv")

df_corrections = pd.read_csv(r"standard_state_correction_factors_20C_120C.csv")

In [3]:
# Helper: look up G for a species at a given temperature
def get_G(df, species_name, temperature_C):
    row = df[(df["species_name"] == species_name) & (df["temperature_C"] == temperature_C)]
    if row.empty:
        raise ValueError(f"Species {species_name!r} not found at {temperature_C} C")
    return row["G_total_wb97mv_plus_Gcorr_plus_SMDTHF_kcal_mol"].values[0]

# With flexible column name
def get_G_PdLArOH(df, species_name, temperature_C, column_name):
        row = df[(df["PdLArOH_species"] == species_name) & (df["temperature_C"] == temperature_C)]
        if row.empty:
            raise ValueError(f"Species {species_name!r} not found at {temperature_C} C")
        return row[column_name].values[0]

def get_G_flexible(df, species_name, temperature_C, column_name):
    row = df[(df["species"] == species_name) & (df["temperature_C"] == temperature_C)]
    if row.empty:
        raise ValueError(f"Species {species_name!r} not found at {temperature_C} C")
    return row[column_name].values[0]

def get_G_reactant9(df_reactant9, species_name, temperature_C):
    row = df_reactant9[(df_reactant9["species"] == species_name) & (df_reactant9["temperature_C"] == temperature_C)]
    if row.empty:
        raise ValueError(f"Species {species_name!r} not found at {temperature_C} C")
    return row["Gtotal_wB97MV_gas_SPE_AIMNet2PD_msRRHO_Gsolv_THF_kcal_mol"].values[0]

def get_G_reactant8(df_reactant8, species_name, temperature_C):
    row = df_reactant8[(df_reactant8["species"] == species_name) & (df_reactant8["temperature_C"] == temperature_C)]
    if row.empty:
        raise ValueError(f"Species {species_name!r} not found at {temperature_C} C")
    return row["Gtotal_wB97MV_gas_SPE_AIMNet2PD_msRRHO_Gsolv_THF_kcal_mol"].values[0] # Before: G_total_wb97mv_plus_Gcorr_plus_SMDTHF_kcal_mol

def get_G_BOH3(df_BOH3, species_name, temperature_C):
    row = df_BOH3[(df_BOH3["species"] == species_name) & (df_BOH3["temperature_C"] == temperature_C)]
    if row.empty:
        raise ValueError(f"Species {species_name!r} not found at {temperature_C} C")
    return row["Gtotal_wB97MV_gas_SPE_AIMNet2PD_msRRHO_Gsolv_THF_kcal_mol"].values[0] 

In [4]:
R   = 8.3145
k_B = 1.380649e-23
h   = 6.62607015e-34

dict_ligand = {
    'P1-L1': 'XPhos',
    'P1-L2': 'SPhos',
    'P1-L3': 'RuPhos',
    'P1-L4': 'AmPhos',
    'P1-L5': 'PCy3',
    'P1-L6': 'PPh3',
    'P1-L7': 'tBu3',
    'P2-L1': 'XPhos',
}

dict_aryl_halide = {
    '3-bromoquinoline': '3_bromoquinoline', 
    '2-chloropyridine': '2_chloropyridine',
    '3-chloropyridine': '3_chloropyridine',
}

ligand_list = ["PCy3", "PPh3", "AmPhos", "Catacxium_A", "RuPhos", "SPhos", "tBu3", "XPhos"]
substrate_list = ["3_bromoquinoline", "2_chloropyridine", "3_chloropyridine"]
temperature_list = [30, 50, 70, 90, 110]
case_list = ["case1", "case2", "case3", "case4"]
dict_tm_case = {
    'case1': ('3_bromoquinoline', 'BOH2Ar_compound_2_as_BOH2_35-dimethylisoxazole-4-boronic_acid', 'BPinAr_compound_2_as_BPin_35-dimethylisoxazole-4-boronate'),
    'case2': ('3_chloropyridine', 'BOH2Ar_compound_2_as_BOH2_35-dimethylisoxazole-4-boronic_acid', 'BPinAr_compound_2_as_BPin_35-dimethylisoxazole-4-boronate'),
    'case3': ('3_chloropyridine', 'BOH2Ar_compound_6_benzofuran-2-boronic_acid', 'BPinAr_compound_6_benzofuran-2-boronate'),
    'case4': ('2_chloropyridine', 'BOH2Ar_compound_9_as_BOH2_1-Boc-pyrrole-2-boronic_acid', 'BPinAr_compound_9_as_BPin_1-Boc-pyrrole-2-boronate'),
}

In [5]:
dict_tm_case['case1'][1]

'BOH2Ar_compound_2_as_BOH2_35-dimethylisoxazole-4-boronic_acid'

In [34]:
# Function for OA barrier
def get_OA_barriers(substrate, ligand, case, T):
    # Get the individual G values
    if substrate == "3_bromoquinoline":
        substrate_capitalized = "3_Bromoquinoline"
    elif substrate == "2_chloropyridine":
        substrate_capitalized = "2_chloropyridine"
        substrate = "2_chloropyridine_concerted"
    else:
        substrate_capitalized = substrate 

    G_PdL       = get_G(df_QM, f"PdL_Mono_{ligand}",                        T)
    G_Substrate = get_G(df_QM, f"Substrate_Reizman_{substrate_capitalized}",         T)
    G_OA_react  = get_G(df_QM, f"OA_reactant_{substrate}_{ligand}",         T)
    G_OA_TS     = get_G(df_QM, f"OA_TS_{substrate}_{ligand}",               T)
    G_OA_prod   = get_G(df_QM, f"OA_product_{substrate}_{ligand}",          T)

    # Barriers
    G_ref_PdL       = G_PdL + G_Substrate
    dG_OA_react_sc1 = G_OA_react - G_ref_PdL
    dG_OA_TS_sc1    = G_OA_TS    - G_ref_PdL
    dG_OA_prod_sc1  = G_OA_prod  - G_ref_PdL
    total_barrier_sc1 = max(dG_OA_TS_sc1, dG_OA_TS_sc1 - dG_OA_react_sc1)
                        
    return total_barrier_sc1

def get_standard_state_correction(df_corrections, temperature_C, delta_n):
    """
    Get the standard state correction for a given temperature and delta_n.
    
    Parameters:
    -----------
    df_corrections : DataFrame
        DataFrame with standard state correction values
    temperature_C : float
        Temperature in Celsius
    delta_n : int
        Change in number of species (n_at_point - n_reference)
    
    Returns:
    --------
    correction : float
        Standard state correction in kcal/mol
    """
    row = df_corrections[df_corrections["temperature_C"] == temperature_C]
    if row.empty:
        raise ValueError(f"Temperature {temperature_C} C not found in correction file")
    
    c_T = row["c_1bar_to_1M_per_molecule_kcal_mol"].values[0]
    
    return delta_n * c_T

# Function for TM barrier
def get_TM_barriers(ligand, case, T, apply_standard_state_correction=True, df_corrections=df_corrections, verbose=False):
    """
    Calculate transmetalation barriers including dimers 8 and 9.

    Standard state correction converts from 1 bar to 1 M.
    Reference state: 2*PdLArOH + ArB(OH)2 (3 species, 1 bar)

    Missing pre-complex species (e.g. TM8 or TM9) are silently skipped;
    the barrier is computed from whichever pre-complexes are available.

    Returns
    -------
    total_barrier : float
        The TM barrier in kcal/mol (TS relative to most stable pre-complex).
    results : dict
        All intermediate relative free energies and metadata.
    """
    # --- Substrate name handling ---
    substrate = dict_tm_case[case][0]
    if substrate == "3_bromoquinoline":
        substrate_capitalized = "3_Bromoquinoline"
        substrate_concerted   = "3_bromoquinoline"
    elif substrate == "2_chloropyridine":
        substrate_capitalized = "2_chloropyridine"
        substrate_concerted   = "2_chloropyridine_concerted"
    else:
        substrate_capitalized = substrate
        substrate_concerted   = substrate

    # --- Helper: return None instead of raising if a species is missing ---
    def try_get_G(fetch_fn, *args):
        try:
            return fetch_fn(*args)
        except Exception:
            return None

    # --- Raw free energies (1 bar) ---
    G_PdLArOH     = get_G(df_QM,       f"PdLArOH_{substrate_concerted}_{ligand}", T)
    G_BOH2Ar      = get_G(df_QM,        dict_tm_case[case][1],                     T)
    G_BOH3        = get_G_BOH3(df_BOH3, "BOH3_neutral",                            T)
    G_TM_react_10 = get_G(df_QM,       f"TM_reactant_{case}_{ligand}_as_10",       T)
    G_TM_react    = get_G(df_QM,       f"TM_reactant_{case}_{ligand}",             T)
    G_TM_TS       = get_G(df_QM,       f"TM_TS_{case}_{ligand}",                   T)
    G_TM_prod     = get_G(df_QM,       f"TM_product_{case}_{ligand}",              T)

    # Optional pre-complexes — may be missing for some ligands/cases
    G_TM_react_9  = try_get_G(get_G_reactant9, df_reactant9, f"TM_reactant_{case}_{ligand}_as_9", T)
    G_TM_react_8  = try_get_G(get_G_reactant8, df_reactant8, f"TM8_OH_{substrate}_{ligand}",      T)

    # --- Reference state: 2*PdLArOH + ArB(OH)2 (n_ref = 3 species) ---
    G_ref = 2 * G_PdLArOH + G_BOH2Ar
    n_ref = 3

    # Species counts at each point along the profile
    n_points = {
        'react_8':  2,   # TM_react_8 + ArB(OH)2
        'react_9':  1,   # TM_react_9 (dimer, all in one)
        'react_10': 2,   # TM_react_10 + PdLArOH
        'react':    2,   # TM_react + PdLArOH
        'TS':       2,   # TS + PdLArOH
        'prod':     3,   # TM_prod + BOH3 + PdLArOH
    }

    # --- Relative energies at 1 bar (None if species was missing) ---
    dG_1bar = {
        'react_8':  (G_TM_react_8  + G_BOH2Ar)           - G_ref  if G_TM_react_8 is not None else None,
        'react_9':   G_TM_react_9                         - G_ref  if G_TM_react_9 is not None else None,
        'react_10': (G_TM_react_10 + G_PdLArOH)          - G_ref,
        'react':    (G_TM_react    + G_PdLArOH)          - G_ref,
        'TS':       (G_TM_TS       + G_PdLArOH)          - G_ref,
        'prod':     (G_TM_prod     + G_BOH3 + G_PdLArOH) - G_ref,
    }

    # --- Standard state correction: 1 bar → 1 M ---
    if apply_standard_state_correction and df_corrections is not None:
        delta_n     = {k: n - n_ref for k, n in n_points.items()}
        corrections = {k: get_standard_state_correction(df_corrections, T, dn)
                       for k, dn in delta_n.items()}
        dG = {k: (dG_1bar[k] + corrections[k] if dG_1bar[k] is not None else None)
              for k in dG_1bar}
    else:
        delta_n     = None
        corrections = None
        dG          = dG_1bar

    # --- Barrier: TS relative to most stable available pre-complex ---
    pre_complex_keys = ['react_8', 'react_9', 'react_10', 'react']
    available_pre_complexes = {k: dG[k] for k in pre_complex_keys if dG[k] is not None}

    if not available_pre_complexes:
        raise ValueError(f"No pre-complex species found for {ligand} / {case} — cannot compute barrier.")

    min_pre_complex     = min(available_pre_complexes.values())
    min_pre_complex_key = min(available_pre_complexes, key=available_pre_complexes.get)
    total_barrier       = dG['TS'] - min_pre_complex

    # --- Optional verbose output ---
    if verbose:
        missing = [k for k in pre_complex_keys if dG[k] is None]
        print(f"TM Barrier Calculation: {substrate} / {ligand} @ {T} °C")
        print("=" * 60)
        print(f"Reference state : 2*PdLArOH + ArB(OH)2 ({n_ref} species, 1 bar)")
        if missing:
            print(f"  [!] Missing pre-complexes (skipped): {', '.join(missing)}")
        if corrections is not None:
            c_per_mol = get_standard_state_correction(df_corrections, T, 1)
            print(f"Std. state corr.: 1 bar → 1 M  |  c({T} °C) = {c_per_mol:.4f} kcal/mol per molecule")
            print(f"  {'Point':<12} {'Δn':>4}  {'correction':>12}  {'ΔG (1 M)':>12}")
            print(f"  {'-'*44}")
            for k in dG:
                if dG[k] is not None:
                    print(f"  {k:<12} {delta_n[k]:>4}  {corrections[k]:>+12.4f}  {dG[k]:>+12.2f}  kcal/mol")
                else:
                    print(f"  {k:<12} {'—':>4}  {'—':>12}  {'missing':>12}")
        else:
            for k, v in dG.items():
                print(f"  dG({k:<10}) : {v:+.2f} kcal/mol" if v is not None else f"  dG({k:<10}) : missing")
        print(f"\n  Most stable pre-complex : {min_pre_complex_key} @ {min_pre_complex:+.2f} kcal/mol")
        print(f"  Total barrier           : {total_barrier:+.2f} kcal/mol")

    return total_barrier, {
        "dG_react_8":          dG['react_8'],
        "dG_react_9":          dG['react_9'],
        "dG_react_10":         dG['react_10'],
        "dG_react":            dG['react'],
        "dG_TS":               dG['TS'],
        "dG_prod":             dG['prod'],
        "min_pre_complex":     min_pre_complex,
        "min_pre_complex_key": min_pre_complex_key,
        "total_barrier":       total_barrier,
        "corrections_applied": corrections is not None,
        "missing_pre_complexes": list(available_pre_complexes.keys()),
        "delta_n":             delta_n,
        "corrections":         corrections,
    }

# Function for absolute OA TS
def get_OA_TS_absolute(substrate, ligand, case, T):
    substrate = dict_tm_case[case][0]
    if substrate == "3_bromoquinoline":
        substrate_capitalized = "3_Bromoquinoline"
    elif substrate == "2_chloropyridine":
        substrate_capitalized = "2_chloropyridine"
        substrate = "2_chloropyridine_concerted"
    else:
        substrate_capitalized = substrate 

    G_OA_TS = get_G(df_QM, f"OA_TS_{substrate}_{ligand}", T)

    return G_OA_TS

# Function for absolute TM TS
def get_TM_TS_absolute(substrate, ligand, case, T):
    substrate = dict_tm_case[case][0]
    if substrate == "3_bromoquinoline":
        substrate_capitalized = "3_Bromoquinoline"
    elif substrate == "2_chloropyridine":
        substrate_capitalized = "2_chloropyridine"
        substrate = "2_chloropyridine_concerted"
    else:
        substrate_capitalized = substrate
    
    G_TM_TS = get_G(df_QM, f"TM_TS_{case}_{ligand}", T)

    return G_TM_TS 

# Function for pinacol boronate dehydration
def get_TM_dehydration_barrier(substrate, ligand, case, T):
    substrate = dict_tm_case[case][0]
    if case == "case3" or case == "case4":
        dG_TM_dehydration = 0
    else:      
        if substrate == "3_bromoquinoline":
            substrate_capitalized = "3_Bromoquinoline"
        elif substrate == "2_chloropyridine":
            substrate_capitalized = "2_chloropyridine"
            substrate = "2_chloropyridine_concerted"
        else:
            substrate_capitalized = substrate 

        # Fetch free energies
        G_PdLArOH  = get_G(df_QM, f"PdLArOH_{substrate}_{ligand}", T)
        G_pinacol = get_G(df_QM, "Pinacol", T)
        G_TM_react = get_G(df_QM, f"TM_reactant_{case}_{ligand}", T) 
        G_Bpin = get_G(df_QM, dict_tm_case[case][2], T)
        G_H2O = get_G(df_QM , "H2O", T)
        G_TM_TS    = get_G(df_QM, f"TM_TS_{case}_{ligand}", T)

        # Reference state
        G_ref_TM_sc2 = G_PdLArOH + G_Bpin + 2*G_H2O   

        dG_TM_react_sc2 = (G_TM_react + G_pinacol) - G_ref_TM_sc2
        dG_TM_TS_sc2    = (G_TM_TS + G_pinacol)    - G_ref_TM_sc2
        dG_TM_dehydration = max(dG_TM_TS_sc2, dG_TM_TS_sc2 - dG_TM_react_sc2)

    return dG_TM_dehydration 

# Function for absolute energy of substrates (arykl halides)
def get_aryl_halide_absolute(substrate, ligand, case, T):
    substrate = dict_tm_case[case][0]
    if substrate == "3_bromoquinoline":
        substrate_capitalized = "3_Bromoquinoline"
    elif substrate == "2_chloropyridine":
        substrate_capitalized = "2_chloropyridine"
        substrate = "2_chloropyridine_concerted"
    else:
        substrate_capitalized = substrate 

    G_Substrate = get_G(df_QM, f"Substrate_Reizman_{substrate_capitalized}", T)

    return G_Substrate

def get_boron_species_absolute(substrate, ligand, case, T):
    boron_species = dict_tm_case[case][1]
    G_boron_species = get_G(df_QM, boron_species, T)

    return G_boron_species

def calc_rate_eyring(deltaG_kcal, T_C):
    T_K = T_C + 273.15
    deltaG_J = deltaG_kcal * 4184.0
    rate = (k_B * T_K / h) * np.exp(-deltaG_J / (R * T_K))
    return rate

# Calculates rate constant of the forward reaction for OA
def get_OA_rate(substrate, ligand, case, T): 
    OA_barrier = get_OA_barriers(substrate, ligand, case, T)
    OA_rate = calc_rate_eyring(OA_barrier, T)
    return OA_rate

# Calculates rate constant of the forward reaction for TM
def get_TM_rate(ligand, case, T, apply_standard_state_correction=True, df_corrections=df_corrections, verbose=False):
    TM_barrier, _ = get_TM_barriers(ligand, case, T, apply_standard_state_correction=apply_standard_state_correction, df_corrections=df_corrections, verbose=verbose)
    TM_rate = calc_rate_eyring(TM_barrier, T)
    return TM_rate

# Calculates the difference in free energy between product and TM10 species
def get_TM10_product_dG(substrate, ligand, case, T):
    substrate = dict_tm_case[case][0]
    if substrate == "3_bromoquinoline":
        substrate_capitalized = "3_Bromoquinoline"
    elif substrate == "2_chloropyridine":
        substrate_capitalized = "2_chloropyridine"
        substrate = "2_chloropyridine_concerted"
    else:
        substrate_capitalized = substrate 

    G_TM_react_10 = get_G(df_QM, f"TM_reactant_{case}_{ligand}_as_10", T)
    G_OA_prod   = get_G(df_QM, f"OA_product_{substrate}_{ligand}", T)

    dG_TM10_product = G_OA_prod - G_TM_react_10

    return dG_TM10_product

def get_dG_PdLArOH(substrate, ligand, case, T):
    substrate = dict_tm_case[case][0]
    if substrate == "3_bromoquinoline":
        substrate_capitalized = "3_Bromoquinoline"
    elif substrate == "2_chloropyridine":
        substrate_capitalized = "2_chloropyridine"
        substrate = "2_chloropyridine_concerted"
    else:
        substrate_capitalized = substrate 

    dG_PdLArOH_X = get_G_PdLArOH(df_ggw_reactions_all_species, f"PdLArOH_{substrate}_{ligand}", T, "deltaG_1M_kcal_mol_wb97mv_SMDTHF")

    return dG_PdLArOH_X

def get_G_precatalyst(substrate, ligand, case, T, cat_gen):
    substrate = dict_tm_case[case][0]
    if substrate == "3_bromoquinoline":
        substrate_capitalized = "3_Bromoquinoline"
    elif substrate == "2_chloropyridine":
        substrate_capitalized = "2_chloropyridine"
        substrate = "2_chloropyridine_concerted"
    else:
        substrate_capitalized = substrate 

    if cat_gen == "G3":
        if ligand == "RuPhos" or ligand == "SPhos":
            G_precatalyst = get_G_flexible(df_precat, f"{cat_gen}_{ligand}_{cat_gen}_OMs_from_G2", T, "Gtotal_wB97MV_gas_SPE_AIMNet2PD_msRRHO_Gsolv_THF_kcal_mol")
        else:
            G_precatalyst = get_G_flexible(df_precat, f"{cat_gen}_Mono_{ligand}_{cat_gen}_OMs_fromG2", T, "Gtotal_wB97MV_gas_SPE_AIMNet2PD_msRRHO_Gsolv_THF_kcal_mol")
    elif cat_gen == "G2":
            G_precatalyst = get_G_flexible(df_precat, f"{cat_gen}_Mono_{ligand}_{cat_gen}_fixedPdN", T, "Gtotal_wB97MV_gas_SPE_AIMNet2PD_msRRHO_Gsolv_THF_kcal_mol")
    
    return G_precatalyst

def get_dG_precatalyst(substrate, ligand, case, T, cat_gen):
    substrate = dict_tm_case[case][0]
    if substrate == "3_bromoquinoline":
        substrate_capitalized = "3_Bromoquinoline"
    elif substrate == "2_chloropyridine":
        substrate_capitalized = "2_chloropyridine"
        substrate = "2_chloropyridine_concerted"
    else:
        substrate_capitalized = substrate
    
    G_precatalyst = get_G_precatalyst(substrate, ligand, case, T, cat_gen)
    G_PdL = get_G(df_QM, f"PdL_Mono_{ligand}", T)
    dG_precatalyst = G_precatalyst - G_PdL # This is with the assumption that the other part of precatalyst is not really needed. For a deeper and more meaningful analysis, it should be included

    return dG_precatalyst




In [37]:
# Testing the functions
d_G_OA_TEST = get_OA_barriers("2_chloropyridine", "RuPhos", "case3", 90)
print(d_G_OA_TEST)
d_G_TM_TEST = get_TM_barriers("PPh3", "case1", 70, apply_standard_state_correction=True, df_corrections=df_corrections, verbose=True)
print(d_G_TM_TEST)
print(f"dG(TM dehydration): {get_TM_dehydration_barrier('3_bromoquinoline', 'AmPhos', 'case3', 110):.2f} kcal/mol")
print(get_OA_TS_absolute("3_chloropyridine", "RuPhos", "case3", 90))
print(get_TM_TS_absolute("3_bromoquinoline", "AmPhos", "case1", 110))
print(f"Kinetics OA: {get_OA_rate('2_chloropyridine', 'RuPhos', 'case3', 90):.2e}")
print(f"Kinetics TM: {get_TM_rate('PPh3', 'case1', 70, apply_standard_state_correction=True, df_corrections=df_corrections, verbose=False):.2e}")  
print(f"dG(TM10 - product): {get_TM10_product_dG('3_bromoquinoline', 'AmPhos', 'case1', 110):.2f} kcal/mol") # It seems that this barrier is not going to work because of the difference in methods used
print(f"dG(PdLArOH): {get_dG_PdLArOH('2_chloropyridine', 'AmPhos', 'case3', 80):.2f} kcal/mol")
print(f"G_precatalyst (G2): {get_G_precatalyst('3_bromoquinoline', 'PPh3', 'case4', 80, 'G2'):.2f} kcal/mol")
print(f"G_precatalyst (G3): {get_G_precatalyst('3_bromoquinoline', 'PPh3', 'case4', 80, 'G3'):.2f} kcal/mol")
print(f"Precatalyst equilibrium: {get_dG_precatalyst('3_bromoquinoline', 'PPh3', 'case4', 80, 'G3'):.2f}")


18.44885087525472
TM Barrier Calculation: 3_bromoquinoline / PPh3 @ 70 °C
Reference state : 2*PdLArOH + ArB(OH)2 (3 species, 1 bar)
Std. state corr.: 1 bar → 1 M  |  c(70 °C) = 2.2851 kcal/mol per molecule
  Point          Δn    correction      ΔG (1 M)
  --------------------------------------------
  react_8        -1       -2.2851        -19.19  kcal/mol
  react_9        -2       -4.5701        -30.67  kcal/mol
  react_10       -1       -2.2851        -12.35  kcal/mol
  react          -1       -2.2851        -11.49  kcal/mol
  TS             -1       -2.2851         -4.58  kcal/mol
  prod            0       +0.0000        -15.32  kcal/mol

  Most stable pre-complex : react_9 @ -30.67 kcal/mol
  Total barrier           : +26.08 kcal/mol
(np.float64(26.080716584802747), {'dG_react_8': np.float64(-19.192073018283725), 'dG_react_9': np.float64(-30.66514368074602), 'dG_react_10': np.float64(-12.35229105781126), 'dG_react': np.float64(-11.485932296135605), 'dG_TS': np.float64(-4.5844270959

In [38]:
df_reizman_to_add = df_reizman.copy()
for index, row in df_reizman_to_add.iterrows():
    substrate = dict_aryl_halide[row["name_aryl_halide"]]
    # print(substrate)
    ligand = dict_ligand[row["catalyst"]]
    # Catalyst generation
    cat_P = row["catalyst"].split("-")[0]
    if cat_P == "P1":
        cat_gen = "G3"
    elif cat_P == "P2":
        cat_gen = "G2"
    # print(ligand)
    case_input = f"case{row['case']}"
    # print(case_input)
    T = round(row["temp_c"], 0)
    # print(T)
    dG_OA = get_OA_barriers(substrate, ligand, case_input, T)
    dG_TM, _ = get_TM_barriers(ligand, case_input, T, apply_standard_state_correction=True, df_corrections=df_corrections, verbose=True)  
    G_OA_TS = get_OA_TS_absolute(substrate, ligand, case_input, T)
    G_TM_TS = get_TM_TS_absolute(substrate, ligand, case_input, T)
    dG_TM_dehydration = get_TM_dehydration_barrier(substrate, ligand, case_input, T)
    G_Substrate = get_aryl_halide_absolute(substrate, ligand, case_input, T)
    G_boron_species = get_boron_species_absolute(substrate, ligand, case_input, T)
    k_OA = get_OA_rate(substrate, ligand, case_input, T)
    k_TM = get_TM_rate(ligand, case_input, T, apply_standard_state_correction=True, df_corrections=df_corrections, verbose=False)
    dG_PdLArOH = get_dG_PdLArOH(substrate, ligand, case_input, T)
    G_precatalyst = get_G_precatalyst(substrate, ligand, case_input, T, cat_gen)
    dG_precatalyst = get_dG_precatalyst(substrate, ligand, case_input, T, cat_gen)
    
    df_reizman_to_add.at[index, 'dG_OA'] = dG_OA
    df_reizman_to_add.at[index, 'dG_TM'] = dG_TM
    df_reizman_to_add.at[index, 'G_OA_TS'] = G_OA_TS
    df_reizman_to_add.at[index, 'G_TM_TS'] = G_TM_TS
    df_reizman_to_add.at[index, 'dG_boron_hydrolysis'] = dG_TM_dehydration
    df_reizman_to_add.at[index, 'G_Substrate'] = G_Substrate
    df_reizman_to_add.at[index, 'G_boron_species'] = G_boron_species
    df_reizman_to_add.at[index, 'k_OA'] = k_OA
    df_reizman_to_add.at[index, 'k_TM'] = k_TM
    df_reizman_to_add.at[index, 'dG_PdLArOH'] = dG_PdLArOH
    df_reizman_to_add.at[index, 'G_precatalyst'] = G_precatalyst
    df_reizman_to_add.at[index, 'dG_precatalyst'] = dG_precatalyst
    print(f"Row {index}: dG_OA = {dG_OA:.2f} kcal/mol, dG_TM = {dG_TM:.2f} kcal/mol",
          f"G_OA_TS = {G_OA_TS:.2f} kcal/mol, G_TM_TS = {G_TM_TS:.2f} kcal/mol",
          f"dG_TM_dehydration = {dG_TM_dehydration:.2f} kcal/mol",
          f"G_Substrate = {G_Substrate:.2f} kcal/mol, G_boron_species = {G_boron_species:.2f} kcal/mol")


TM Barrier Calculation: 3_bromoquinoline / RuPhos @ 30.0 °C
Reference state : 2*PdLArOH + ArB(OH)2 (3 species, 1 bar)
  [!] Missing pre-complexes (skipped): react_8
Std. state corr.: 1 bar → 1 M  |  c(30.0 °C) = 1.9440 kcal/mol per molecule
  Point          Δn    correction      ΔG (1 M)
  --------------------------------------------
  react_8         —             —       missing
  react_9        -2       -3.8881         -8.62  kcal/mol
  react_10       -1       -1.9440        -11.32  kcal/mol
  react          -1       -1.9440         +3.26  kcal/mol
  TS             -1       -1.9440         +8.78  kcal/mol
  prod            0       +0.0000        -15.47  kcal/mol

  Most stable pre-complex : react_10 @ -11.32 kcal/mol
  Total barrier           : +20.09 kcal/mol
Row 0: dG_OA = 19.21 kcal/mol, dG_TM = 20.09 kcal/mol G_OA_TS = -2989069.91 kcal/mol, G_TM_TS = -1735596.76 kcal/mol dG_TM_dehydration = 25.59 kcal/mol G_Substrate = -1867009.32 kcal/mol, G_boron_species = -314173.30 kcal/mol


In [ ]:
df_reizman_to_add.to_excel("Reizman_database - MECH-ML-APPEND - DIMERS.xlsx", index=False)